# New columns and sorting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/y-rosenthal/yr-pre-ml/blob/main/jb/chapters/11-new-columns-and-sorting.ipynb)

A data file rarely has every column that we need. The sales file has the units, the price and the discount of
each sale, but not the money that the sale brought in. In this chapter, we compute new columns from the old ones,
tidy up the column names, and sort the rows to answer questions such as "Which were our five biggest sales?"

**In this chapter, you will learn to:**

- add a column computed from other columns, and round it;
- make a column from a part of a text column, with `.str[:7]`;
- make a column from a condition, with the `loc` rule;
- rename and drop columns;
- sort the rows by one or two columns, find the top rows, give them a fresh index, and use `nlargest`.

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/y-rosenthal/yr-pre-ml/main/data/"

We use the sales of a store chain again: 2,000 sales in one year, one row for each sale.

In [ ]:
sales = pd.read_csv(DATA_URL + "sales.csv")
sales.head()

## A new column from other columns

The **revenue** of a sale is the money it brought in: the units, times the price, minus the discount. A discount
of 0.1 means 10% off, so the customer pays `1 - 0.1 = 0.9` of the price. We compute it for every row at once, and
store it in a new column:

In [ ]:
sales["revenue"] = sales["units"] * sales["price"] * (1 - sales["discount"])
sales[["units", "price", "discount", "revenue"]].head()

The new column `revenue` appears at the right end of the table. Check row 0: 3 units at $46.99 with a discount of
0.1 give 3 × 46.99 × 0.9 = 126.873.

The arithmetic works row by row, as it does for arrays of the same length: pandas multiplies the units of row 0 by
the price of row 0, and so on, for all 2,000 rows, with no loop. The rule:

- `df["new"] = ...` with a new name **adds** a column;
- with the name of a column that exists, it **replaces** that column.

## Rounding a column

Money has two decimal places. `round(2)` rounds every value of a column, and we put the result back in the same
column:

In [ ]:
sales["revenue"] = sales["revenue"].round(2)
sales[["units", "price", "discount", "revenue"]].head()

126.873 became 126.87, and 41.496 (row 4) became 41.50. The name `revenue` is on both sides of `=`, so the
rounded column replaces the old one.

## A column from text: `.str`

The column `date` holds text such as `"2025-03-14"`. To compare months, we want just the year and the month,
`"2025-03"`: the first seven characters. For one Python string, we would write `date[:7]`. For a whole column, we
put `.str` before the slice:

In [ ]:
sales["month"] = sales["date"].str[:7]
sales[["date", "month"]].head()

`.str` tells pandas to do a text operation on **each** value of the column, here the slice `[:7]`. Every date of
January now has the month `2025-01`. (pandas has many other text operations after `.str`; we need only this one.)

> **Try it:** `sales["month"].nunique()` counts the different months. Is the answer what you expect?

## A column from a condition

Let's label each sale "small" or "large": large when it has 8 units or more. We do it in two steps. First, every row
gets `"small"`: one value on the right of `=` fills the whole column. Then the `loc` rule of the chapter
*Filtering rows* changes the rows that match the condition:

In [ ]:
sales["size"] = "small"
sales.loc[sales["units"] >= 8, "size"] = "large"
sales[["units", "size"]].head()

Row 0 has 3 units, so it stays `small`. The rows 1 to 4 have 8 or more units, so they became `large`.

> **Try it:** count the large sales with `(sales["size"] == "large").sum()`. What share of the 2,000 sales is
> that?

## Renaming columns

The name `size` is short, but it is not very clear, and a DataFrame also has an attribute `sales.size` (the number
of values in the table, as for arrays). Let's call the column `order_size`. `rename` takes a dictionary from the
old names to the new names:

In [ ]:
sales = sales.rename(columns={"size": "order_size"})
list(sales.columns)

The column is now called `order_size`. Like most pandas methods, `rename` does not change the table itself: it
returns a new one, so we assign it back to `sales`. `list(...)` turns the column names into a plain Python list,
which is easy to read.

The dictionary can hold several names at once, for example `{"units": "quantity", "category": "product_type"}`.

## Dropping columns

Our table now has 10 columns. For a report about revenue, we do not need all of them. `drop(columns=[...])`
returns the table without the columns in the list:

In [ ]:
report = sales.drop(columns=["date", "region", "price", "discount"])
report.head()

`report` has six columns: `store`, `category`, `units`, `revenue`, `month` and `order_size`. The table `sales`
still has all ten, because we assigned the result to a new name.

> **Common mistake:** writing `sales.drop(columns=["date"])` alone on a line and expecting `sales` to change. The
> line shows a table without `date`, but `sales` keeps it. To keep the change, assign it:
> `sales = sales.drop(columns=["date"])`.

## Sorting the rows

`sort_values("revenue")` returns the rows in order of their revenue, from the smallest:

In [ ]:
report.sort_values("revenue").head()

The smallest sale brought in $2.84: one Grocery unit. Look at the index: 110, 1080, 815, ... Each row kept its
label, as we saw in the chapter *Selecting with loc and iloc*.

## The top 5

For the biggest sales first, add `ascending=False`. Then `head()` gives the top 5:

In [ ]:
report.sort_values("revenue", ascending=False).head()

The five biggest sales were all Electronics sales of 3 units, between about $1,374 and $1,698. Note their
`order_size`: our rule called them `small`, because it looked at the units, not the money. Which column we sort by
decides what "big" means.

This "sort, then `head()`" pattern answers many business questions: the top 5 customers, the 10 slowest
deliveries, the cheapest products.

## Sorting by two columns

To sort by two columns, give a list of names. The rows are sorted by the first column, and rows with the same value
there are sorted by the second. `ascending` can take a list too, one `True` or `False` for each column:

In [ ]:
report.sort_values(["store", "revenue"], ascending=[True, False]).head()

The stores are in order from `S1`, and within store S1, the biggest sale comes first. So the first row is the
biggest sale of store S1: $1,514.97.

## A fresh index: `reset_index`

After sorting, the labels are mixed up. For a report, the labels 0, 1, 2, ... often read better.
`reset_index(drop=True)` gives the rows new labels from 0, and throws away the old ones:

In [ ]:
top5 = report.sort_values("revenue", ascending=False).head()
top5 = top5.reset_index(drop=True)
top5

The same five sales, now labelled 0 to 4. Without `drop=True`, pandas would keep the old labels in a new column
named `index`.

## A shortcut: `nlargest`

"Sort from the largest, then take the first rows" is so common that pandas has a shortcut, `nlargest`:

In [ ]:
report.nlargest(5, "revenue")

The same five sales as the top 5 above, with their original labels. `nlargest(5, "revenue")` reads "the 5 rows
with the largest revenue". `nsmallest` does the opposite.

## Summary

- `df["new"] = ...` adds a column (or replaces one with that name):
  `sales["revenue"] = sales["units"] * sales["price"] * (1 - sales["discount"])`.
- `sales["revenue"].round(2)` rounds every value; assign it back to keep it.
- `sales["date"].str[:7]` takes the first seven characters of each text value.
- A column from a condition: `sales["size"] = "small"`, then `sales.loc[sales["units"] >= 8, "size"] = "large"`.
- `df.rename(columns={"old": "new"})` and `df.drop(columns=[...])` return a new table; assign the result.
- `df.sort_values("col")` sorts from the smallest; `ascending=False` from the largest; a list of columns sorts by
  several. `head()` after sorting gives a top 5.
- `reset_index(drop=True)` gives the labels 0, 1, 2, ...; `df.nlargest(5, "col")` gives the 5 largest rows.

## Practice questions

### Question 1 (Easy)

Add a column `price_after_discount`: the price of one unit after the discount, rounded to two decimals. Show the
first rows of `price`, `discount` and `price_after_discount`.

<details class="yr-answer">
<summary>Answer</summary>

The customer pays `1 - discount` of the price. Row 0: 46.99 × 0.9 = 42.29 (rounded).

```python
sales["price_after_discount"] = (sales["price"] * (1 - sales["discount"])).round(2)
sales[["price", "discount", "price_after_discount"]].head()
```

</details>

In [ ]:
# Write your answer here

### Question 2 (Easy)

Which three sales brought in the least revenue? Show them, using `report`.

<details class="yr-answer">
<summary>Answer</summary>

Sort from the smallest (the default), then take the first three rows. They are all Grocery sales of one or two
units.

```python
report.sort_values("revenue").head(3)
```

</details>

In [ ]:
# Write your answer here

### Question 3 (Easy)

Add a column `day` with the day of the month: the last two characters of `date`. Show `date` and `day` for the
first rows.

<details class="yr-answer">
<summary>Answer</summary>

In `"2025-03-14"`, the day is at the positions 8 and 9, so `.str[8:]` takes it. `.str[-2:]` (the last two
characters) works too.

```python
sales["day"] = sales["date"].str[8:]
sales[["date", "day"]].head()
```

</details>

In [ ]:
# Write your answer here

### Question 4 (Medium)

Add a column `discounted` that is `"yes"` for the sales with a discount and `"no"` for the others. How many sales
had a discount?

<details class="yr-answer">
<summary>Answer</summary>

Fill the column with `"no"`, then change the matching rows with `loc`. 982 sales had a discount.

```python
sales["discounted"] = "no"
sales.loc[sales["discount"] > 0, "discounted"] = "yes"
(sales["discounted"] == "yes").sum()
```

</details>

In [ ]:
# Write your answer here

### Question 5 (Medium)

Show the three biggest sales (by revenue) of store S5. Use `nlargest`.

<details class="yr-answer">
<summary>Answer</summary>

Keep the sales of store S5 first (the chapter *Filtering rows*), then take the three largest revenues.

```python
s5 = report[report["store"] == "S5"]
s5.nlargest(3, "revenue")
```

</details>

In [ ]:
# Write your answer here

### Question 6 (Hard)

Make a table of the five biggest **Grocery** sales by revenue, with the labels 0 to 4.

<details class="yr-answer">
<summary>Answer</summary>

Filter, sort from the largest, take five rows, and reset the index. Each step works on the result of the step
before.

```python
grocery = report[report["category"] == "Grocery"]
top_grocery = grocery.sort_values("revenue", ascending=False).head()
top_grocery = top_grocery.reset_index(drop=True)
top_grocery
```

</details>

In [ ]:
# Write your answer here

### Question 7 (Hard)

Sort `report` by month (from January) and, within each month, by revenue from the largest. Show the first three
rows. What do they tell you?

<details class="yr-answer">
<summary>Answer</summary>

Two columns, with a list for `ascending`. The first three rows are the three biggest sales of January. The text
months such as `"2025-01"` sort in time order, because the year comes first and the month has two digits.

```python
report.sort_values(["month", "revenue"], ascending=[True, False]).head(3)
```

</details>

In [ ]:
# Write your answer here

### Question 8 (Challenge)

Replace the two sizes with three: `"small"` for fewer than 4 units, `"medium"` for 4 to 7 units, and `"large"` for
8 units or more. Put them in a new column `size3` and count the sales of each size.

<details class="yr-answer">
<summary>Answer</summary>

Start with `"small"` everywhere, then use `loc` once for each other size. The medium sales need two conditions.
Most sales are small: 900 small, 785 medium and 315 large.

```python
sales["size3"] = "small"
sales.loc[(sales["units"] >= 4) & (sales["units"] <= 7), "size3"] = "medium"
sales.loc[sales["units"] >= 8, "size3"] = "large"
for size in ["small", "medium", "large"]:
    print(size, (sales["size3"] == size).sum())
```

</details>

In [ ]:
# Write your answer here